# Import libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sqlalchemy import create_engine

from underthesea import word_tokenize
import unicodedata
import re


from collections import defaultdict

# Connect to workbench

In [2]:
username = 'root'
password = '08042004'
host = 'localhost'
port = '3306'
database = 'law_db'

# Tạo engine kết nối đến MySQL
engine = create_engine(
    f"mysql+mysqlconnector://{username}:{password}@{host}:{port}/{database}?charset=utf8mb4"
)


# Lấy dữ liệu 

In [3]:
# Lấy dữ liệu
df_case = pd.read_sql("SELECT id, case_level,text, url, file FROM `case`", con = engine)

In [4]:
df_case.head()

,id,case_level,text,url,file
0,1,Sơ thẩm,...,https://congbobanan.toaan.gov.vn/2ta737426t1cv...,https://congbobanan.toaan.gov.vn/5ta737426t1cv...
1,2,Sơ thẩm,1 \n \nTÒA ÁN NHÂN DÂN \nCỘNG HÒA XÃ HỘI CHỦ N...,https://congbobanan.toaan.gov.vn/2ta162985t1cv...,https://congbobanan.toaan.gov.vn/5ta162985t1cv...
2,3,Sơ thẩm,\n1 \nTÒA ÁN NHÂN DÂN QUẬN \nLONG BIÊN – TP H...,https://congbobanan.toaan.gov.vn/2ta162882t1cv...,https://congbobanan.toaan.gov.vn/5ta162882t1cv...
3,4,Phúc thẩm,TÒA ÁN NHÂN DÂN \nCỘNG HÒA XÃ HỘI CHỦ N...,https://congbobanan.toaan.gov.vn/2ta158647t1cv...,https://congbobanan.toaan.gov.vn/5ta158647t1cv...
4,5,Sơ thẩm,2 \nTÒA ÁN NHÂN DÂN \nHUYỆN QUAN SƠN \nTỈNH TH...,https://congbobanan.toaan.gov.vn/2ta141585t1cv...,https://congbobanan.toaan.gov.vn/5ta141585t1cv...


# I. Chuẩn bị dữ liệu

## Hàm tiền xử lý dữ liệu

In [5]:
def preprocess_text(text):
    # text k la str => rỗng
    if not isinstance(text, str):
        return ""
    # chữ thường
    # text = text.lower()

    # chuyển từ viết tắt 
    abbrList = {
        'blhs': 'bộ luật hình sự',
        'tths': 'tố tụng hình sự',
        'thhs': 'thi hành hình sự',
        'blds': 'bộ luật dân sự',
        'ttds': 'tố tụng dân sự',
        'thds': 'thi hành dân sự'
    }

    for abbr, full_form in abbrList.items():
        text = text.replace(abbr, full_form)
    

    text = unicodedata.normalize("NFC", text) # chuẩn hóa mã hóa Unicode > đồng nhất
    # text = re.sub(r"[^a-zA-ZÀ-Ỹà-ỹ0-9\s]", " ", text) #giữ chữ, số , loại kí tự đbiet
    
    
    text = text.replace('\n', ' ')   # thay \n bằng khoảng trắng
    # khoảng trắng thừa = 1 space 
    while '  ' in text:
        text = text.replace('  ', ' ')
    text = text.strip() # xoa space đầu cuối 


    return text


In [6]:
# test 
text = "Blhs và tths quy định thế nào? Thhs, blds, ttds, thds liên quan. blhs.                          abc"
result = preprocess_text(text)
print(result)

Blhs và tố tụng hình sự quy định thế nào? Thhs, bộ luật dân sự, tố tụng dân sự, thi hành dân sự liên quan. bộ luật hình sự. abc


In [7]:
import json

# Chọn cột cần thiết
df_filtered = df_case[['id', 'case_level', 'text']].copy()

# Xử lý text
df_filtered['text'] = df_filtered['text'].apply(preprocess_text)
print(df_filtered)
# # Giới hạn id (ví dụ <= 1006)
# max_id = 2000
# min_id = 900
# df_filtered = df_filtered[(df_filtered['id'] >= min_id) & (df_filtered['id'] <= max_id)]


# # Convert thành list of dicts
# data = df_filtered.to_dict(orient="records")

# # Xuất JSON

# with open(f"cases3.json", "w", encoding="utf-8") as f:
#     json.dump(data, f, ensure_ascii=False, indent=4)



          id case_level                                               text
0          1    Sơ thẩm  TÒA ÁN NHÂN DÂN QUẬN NGÔ QUYỀN THÀNH PHỐ HẢI P...
1          2    Sơ thẩm  1 TÒA ÁN NHÂN DÂN CỘNG HÒA XÃ HỘI CHỦ NGHĨA VI...
2          3    Sơ thẩm  1 TÒA ÁN NHÂN DÂN QUẬN LONG BIÊN – TP HÀ NỘI –...
3          4  Phúc thẩm  TÒA ÁN NHÂN DÂN CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT...
4          5    Sơ thẩm  2 TÒA ÁN NHÂN DÂN HUYỆN QUAN SƠN TỈNH THANH HÓ...
...      ...        ...                                                ...
9996   10218    Sơ thẩm  1 TÒA ÁN NHÂN DÂN THÀNH PHỐ CAO BẰNG TỈNH CAO ...
9997   10219    Sơ thẩm  TÒA ÁN NHÂN DÂN CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT...
9998   10220    Sơ thẩm  TÒA ÁN NHÂN DÂN CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT...
9999   10221    Sơ thẩm  1 TÒA ÁN NHÂN DÂN HUYỆN HƯNG HÀ TỈNH THÁI BÌNH...
10000  10222    Sơ thẩm  TÒA ÁN NHÂN DÂN CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT...

[10001 rows x 3 columns]


In [8]:
# Lọc các dòng có case_level = "Phúc thẩm"
# Giới hạn id (ví dụ <= 1006)
max_id = 3000
min_id = 2000
df_filtered = df_filtered[(df_filtered['id'] >= min_id) & (df_filtered['id'] <= max_id)]
df_phuc = df_filtered[df_filtered['case_level'] == "Phúc thẩm"]

results = []

for _, row in df_phuc.iterrows():
    case_id = row['id']
    case_level = row['case_level']
    text = row['text']
    
    # Regex: tìm "sơ thẩm số ..." đến dấu chấm đầu tiên sau đó
    match = re.search(r"(sơ thẩm số.*?\.)", text, flags=re.IGNORECASE | re.DOTALL)
    
    if match:
        sentence = match.group(1).strip()
    else:
        sentence = "Không tìm thấy thông tin sơ thẩm số"
    
    # Đưa vào danh sách kết quả
    results.append(f"{case_id}\n{case_level}\n{sentence}\n")
from datetime import datetime

# Lấy thời gian hiện tại
now = datetime.now()

# Xuất ra string có cả mili giây
time_str = now.strftime("%Y%m%d_%H%M%S") 
print(time_str)
# Xuất file txt
with open(f"output{min_id}-{max_id}.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(results))

20250826_015419
